In [1]:
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

# 下載繁體中文字型
!wget -O SourceHanSerifTW-VF.ttf https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf

# 加入字型檔
fm.fontManager.addfont('SourceHanSerifTW-VF.ttf')

# 設定字型
#
mpl.rc('font', family='Source Han Serif TW VF')

--2026-06-18 03:04:52--  https://github.com/adobe-fonts/source-han-serif/raw/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/adobe-fonts/source-han-serif/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf [following]
--2026-06-18 03:04:53--  https://raw.githubusercontent.com/adobe-fonts/source-han-serif/release/Variable/TTF/Subset/SourceHanSerifTW-VF.ttf
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16851180 (16M) [application/octet-stream]
Saving to: ‘SourceHanSerifTW-VF.ttf’

SourceHanSerifTW-VF 100%[===================>]  16.07M  94.5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# ── 中文字型（Colab 請先執行以下安裝）──────────────
# !apt-get -qq install fonts-noto-cjk

# fonts = fm.findSystemFonts()
# noto = [f for f in fonts if 'NotoSansCJK' in f or 'noto' in f.lower()]
# if noto:
#     fm.fontManager.addfont(noto[0])
#     prop = fm.FontProperties(fname=noto[0])
#     plt.rcParams['font.family'] = prop.get_name()
# plt.rcParams['axes.unicode_minus'] = False

# ── 讀取 xlsx ─────────────────────────────────────
df = pd.read_excel(
    "assetAllocation.xlsx",   # ← 上傳到 Colab 後與此腳本同目錄
    sheet_name="0A_原始資料",
    header=0
)
df.columns = ["年份", "SP500", "IEF"]
df = df[df["SP500"] != "-"].copy()
df["SP500"] = pd.to_numeric(df["SP500"]) * 100   # 轉換成 %
df["IEF"]   = pd.to_numeric(df["IEF"])   * 100
df["年份"]  = df["年份"].astype(int)
df = df.reset_index(drop=True)

# ── 繪圖 ──────────────────────────────────────────
years = df["年份"].values
x = np.arange(len(years))
w = 0.38   # 每根柱子寬度

fig, ax = plt.subplots(figsize=(18, 8))
fig.patch.set_facecolor("#f8f9fa")
ax.set_facecolor("#ffffff")

ax.bar(x - w/2, df["SP500"], width=w, color="#2563eb", alpha=0.88,
       label="標普500 (S&P 500)", zorder=3)
ax.bar(x + w/2, df["IEF"],   width=w, color="#b45309", alpha=0.88,
       label="IEF 美債7-10年", zorder=3)

# ── 數值標籤（絕對值大才顯示，避免太擁擠）─────────
for i, (s, b) in enumerate(zip(df["SP500"], df["IEF"])):
    if abs(s) > 8:
        ax.text(x[i] - w/2, s + (0.5 if s >= 0 else -1.5), f"{s:.1f}%",
                ha="center", va="bottom" if s >= 0 else "top",
                fontsize=7, color="#1e40af", fontweight="bold")
    if abs(b) > 6:
        ax.text(x[i] + w/2, b + (0.5 if b >= 0 else -1.5), f"{b:.1f}%",
                ha="center", va="bottom" if b >= 0 else "top",
                fontsize=7, color="#92400e", fontweight="bold")

# ── 危機年份標記 ───────────────────────────────────
crisis = {2002: "科技泡沫", 2008: "金融海嘯", 2022: "升息衝擊"}
for yr, label in crisis.items():
    if yr in list(years):
        xi = list(years).index(yr)
        ax.axvspan(xi - 0.55, xi + 0.55, color="#fee2e2", alpha=0.5, zorder=1)
        ax.text(xi, -42, label, ha="center", va="bottom",
                fontsize=8, color="#991b1b", fontweight="bold")

# ── 格式設定 ───────────────────────────────────────
ax.axhline(0, color="#374151", lw=1.2, zorder=4)
ax.set_xticks(x)
ax.set_xticklabels(years, rotation=45, ha="right", fontsize=10)
ax.set_ylabel("年報酬率 (%)", fontsize=12)
ax.set_xlabel("年份", fontsize=12)
ax.set_title("標普500 vs IEF 美債7-10年　年度報酬率（2000–2024）",
             fontsize=14, fontweight="bold", pad=15)
ax.legend(fontsize=11, loc="upper left")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0f}%"))
ax.set_ylim(-45, 42)
ax.grid(axis="y", alpha=0.3, color="#9ca3af", linestyle="--")
ax.set_xlim(-0.7, len(years) - 0.3)

plt.tight_layout()
plt.savefig("spy_ief_annual_returns.png", dpi=160, bbox_inches="tight")
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

# ── 1. 讀取資料 ──────────────────────────────────
df = pd.read_excel(
    "assetAllocation.xlsx",   # 先把檔案上傳到 Colab
    sheet_name="0A_原始資料",
    header=0
)
df.columns = ["年份", "SP500", "IEF"]
df = df[df["SP500"] != "-"].copy()
df["SP500"] = pd.to_numeric(df["SP500"])
df["IEF"]   = pd.to_numeric(df["IEF"])

returns = df[["SP500", "IEF"]].values
ASSETS  = ["S&P 500", "IEF (中期公債)"]
mu  = returns.mean(axis=0)      # 歷史平均年報酬
cov = np.cov(returns.T)         # 共變異數矩陣
n   = len(mu)

# ── 2. Monte Carlo 隨機組合（背景散點）──────────
np.random.seed(42)
w_mc     = np.random.dirichlet(np.ones(n), 12000)
port_ret = w_mc @ mu
port_vol = np.sqrt(np.einsum("ij,jk,ik->i", w_mc, cov, w_mc))
port_sharpe = port_ret / port_vol

# ── 3. 最佳化效率前緣 ────────────────────────────
target_rets = np.linspace(mu.min() * 0.98, mu.max() * 1.01, 400)
ef_vols, ef_rets_out, ef_wts = [], [], []

for tr in target_rets:
    res = minimize(
        lambda w: np.sqrt(w @ cov @ w),
        x0=np.ones(n) / n, method="SLSQP",
        bounds=[(0, 1)] * n,
        constraints=[
            {"type": "eq", "fun": lambda w: w.sum() - 1},
            {"type": "eq", "fun": lambda w, r=tr: w @ mu - r},
        ],
        options={"ftol": 1e-14, "maxiter": 2000},
    )
    if res.success:
        ef_vols.append(res.fun)
        ef_rets_out.append(tr)
        ef_wts.append(res.x)

ef_vols     = np.array(ef_vols)
ef_rets_out = np.array(ef_rets_out)
ef_wts      = np.array(ef_wts)
sharpe_ef   = ef_rets_out / ef_vols

idx_minvol = np.argmin(ef_vols)
idx_sharpe = np.argmax(sharpe_ef)

# ── 4. 繪圖 ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor("#0f1117")
for ax in axes:
    ax.set_facecolor("#1a1d27")
    ax.tick_params(colors="#cccccc")
    ax.xaxis.label.set_color("#cccccc")
    ax.yaxis.label.set_color("#cccccc")
    ax.title.set_color("#ffffff")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333344")

# 左圖：效率前緣
ax = axes[0]
sc = ax.scatter(port_vol, port_ret, c=port_sharpe, cmap="plasma",
                alpha=0.35, s=6, linewidths=0)
cbar = fig.colorbar(sc, ax=ax, pad=0.01)
cbar.set_label("Sharpe Ratio (rf=0)", color="#cccccc", fontsize=10)
plt.setp(cbar.ax.yaxis.get_ticklabels(), color="#cccccc")

ax.plot(ef_vols, ef_rets_out, color="#00d4ff", lw=2.5, label="效率前緣")
ax.scatter(ef_vols[idx_minvol], ef_rets_out[idx_minvol],
           color="#00ff88", s=140, marker="*", zorder=5, label="最小波動率")
ax.annotate(
    f"Min Vol\nvol={ef_vols[idx_minvol]:.1%}  ret={ef_rets_out[idx_minvol]:.1%}\n"
    f"SP500={ef_wts[idx_minvol,0]:.0%}  IEF={ef_wts[idx_minvol,1]:.0%}",
    xy=(ef_vols[idx_minvol], ef_rets_out[idx_minvol]),
    xytext=(ef_vols[idx_minvol]+0.01, ef_rets_out[idx_minvol]-0.012),
    color="#00ff88", fontsize=8.5,
    arrowprops=dict(arrowstyle="->", color="#00ff88", lw=1),
)
ax.scatter(ef_vols[idx_sharpe], ef_rets_out[idx_sharpe],
           color="#ffcc00", s=140, marker="D", zorder=5, label="最大夏普比率")
ax.annotate(
    f"Max Sharpe={sharpe_ef[idx_sharpe]:.2f}\n"
    f"vol={ef_vols[idx_sharpe]:.1%}  ret={ef_rets_out[idx_sharpe]:.1%}\n"
    f"SP500={ef_wts[idx_sharpe,0]:.0%}  IEF={ef_wts[idx_sharpe,1]:.0%}",
    xy=(ef_vols[idx_sharpe], ef_rets_out[idx_sharpe]),
    xytext=(ef_vols[idx_sharpe]+0.01, ef_rets_out[idx_sharpe]+0.005),
    color="#ffcc00", fontsize=8.5,
    arrowprops=dict(arrowstyle="->", color="#ffcc00", lw=1),
)
for av, ar, ac, name in zip(np.sqrt(np.diag(cov)), mu,
                             ["#ff6b6b","#74b9ff"], ASSETS):
    ax.scatter(av, ar, color=ac, s=100, zorder=5)
    ax.annotate(name, xy=(av, ar), xytext=(av+0.005, ar+0.003),
                color=ac, fontsize=9)

ax.set_xlabel("年化波動率（標準差）", fontsize=11)
ax.set_ylabel("年化平均報酬", fontsize=11)
ax.set_title("效率前緣曲線（2000–2024）", fontsize=13, fontweight="bold")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{x:.0%}"))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f"{y:.0%}"))
ax.legend(fontsize=9, loc="upper left",
          facecolor="#252836", edgecolor="#444", labelcolor="white")
ax.grid(alpha=0.15, color="#888")

# 右圖：最佳權重分配
ax2 = axes[1]
ax2.stackplot(ef_rets_out*100, ef_wts[:,0]*100, ef_wts[:,1]*100,
              labels=ASSETS, colors=["#ff6b6b","#74b9ff"], alpha=0.85)
ax2.axvline(ef_rets_out[idx_minvol]*100, color="#00ff88", lw=1.2, ls="--",
            label=f"最小波動率 ({ef_rets_out[idx_minvol]:.1%})")
ax2.axvline(ef_rets_out[idx_sharpe]*100, color="#ffcc00", lw=1.2, ls="--",
            label=f"最大夏普比率 ({ef_rets_out[idx_sharpe]:.1%})")
ax2.set_xlabel("目標年化報酬 (%)", fontsize=11)
ax2.set_ylabel("配置權重 (%)", fontsize=11)
ax2.set_title("效率前緣各點的最佳權重分配", fontsize=13, fontweight="bold")
ax2.set_xlim(ef_rets_out.min()*100, ef_rets_out.max()*100)
ax2.set_ylim(0, 100)
ax2.legend(fontsize=9, loc="upper left",
           facecolor="#252836", edgecolor="#444", labelcolor="white")
ax2.grid(alpha=0.15, color="#888")

plt.tight_layout(pad=2)
plt.savefig("efficient_frontier.png", dpi=160, bbox_inches="tight", facecolor="#0f1117")
plt.show()